In [7]:
import re
import pandas as pd
from tqdm import tqdm
tqdm.pandas()

In [10]:
# concat gold label data
strong = pd.read_csv('~/code/dynamics-paper/mindset-strong-gender-annot.csv')
strong.head()

,TID,text,created_utc,age,gender,matched_excerpt,user_id
0,822003|adhd|s22moj,stupid 1st world stuff here but putting it for...,1.641981e+09,32.0,female,32f,822003
1,791323|adhd|nbijqe,i am 28f but was born with or developed severa...,1.620918e+09,28.0,female,28f,791323
2,318905|adhd|glrhhf,long post. . . tl dr is my sis wasn't there fo...,1.589764e+09,34.0,female,34f,318905
3,516112|adhd|kezfv4e,34 f here. never wanted kids. there was a peri...,1.703598e+09,34.0,female,34 f,516112
4,692720|adhd|h7dzak3,thank you sooo much this looks great! ! and on...,1.627874e+09,32.0,female,32f,692720


In [9]:
cssrs = pd.read_csv('data/standardized/cssrs_standardized.csv')
mindset = pd.read_csv('data/standardized/mindset_standardized.csv')
umd = pd.read_csv('data/standardized/umd_standardized.csv')

cssrs['dataset'] = 'cssrs'
mindset['dataset'] = 'mindset'
umd['dataset'] = 'umd'

df = pd.concat([cssrs, mindset, umd], ignore_index=True)
df['text'] = df['sentence'].fillna('')

# ── Fix CSSRS list-format posts: "['post1', 'post2', ...]" → joined text ────
list_mask = df['text'].str.startswith("[\'")
if list_mask.any():
    def clean_list_text(s):
        inner = s[2:-2]                # strip outer [' and ']
        parts = inner.split("', '")    # split on delimiter
        return ' '.join(parts)
    df.loc[list_mask, 'text'] = df.loc[list_mask, 'text'].apply(clean_list_text)
    print(f"Fixed {list_mask.sum()} CSSRS list-format posts → joined plain text")

df['binary_label'] = df['label'].apply(lambda x: 1 if x in ['suicidal', 'at_risk', 'has_condition'] else 0)

print(f"Total samples: {len(df)}")
print(f"Datasets: {df['dataset'].value_counts().to_dict()}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/standardized/cssrs_standardized.csv'

In [21]:
# ================================================================
# GENDER DETECTION PATTERNS - 7 tiers, ordered by confidence
# ================================================================

# --- Tier 1: AGE + GENDER combos (Reddit convention, highest conf) ---
AGE_GENDER = [
    # (25F)  [30M]  (25 female)
    re.compile(r'[\(\[]\s*(\d{1,3})\s*(m|f|nb|male|female|nonbinary|enby)\s*[\)\]]', re.I),
    # F(25)  male(30)
    re.compile(r'\b(m|f|male|female|nb|nonbinary|enby)\s*[\(\[]\s*(\d{1,3})\s*[\)\]]', re.I),
    # 25/F  30/M  25/female
    re.compile(r'\b(\d{1,3})\s*/\s*(m|f|nb|male|female|nonbinary|enby)\b', re.I),
    # 25f  30m  (adjacent, no separator)
    re.compile(r'(?<![.\d/])\b(\d{1,3})\s*(m|f|nb|male|female|nonbinary|enby)\b', re.I),
    # 25 year old female / 30yo male
    re.compile(
        r'\b(\d{1,3})\s*(?:yo|y/?o|years?\s*old|yrs?\s*old)\s+'
        r'(male|female|man|woman|boy|girl|guy|gal|lady|dude|m|f)\b', re.I),
    # I'm a 25 year old female
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?(\d{1,3})\s*"
        r"(?:yo|y/?o|years?\s*old|yrs?\s*old)\s+"
        r"(male|female|man|woman|boy|girl|guy|gal|lady|dude|m|f)\b", re.I),
]

# --- Tier 2: EXPLICIT self-identification ---------------------------------
SELF_ID = [
    # I am [a] girl/guy/man/woman/male/female/...
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?"
        r"(girl|boy|man|woman|male|female|guy|gal|lady|dude|gentleman"
        r"|nonbinary|nb|enby)\b", re.I),
    # I am [a] [adjective] girl/guy/...
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?"
        r"(?:young|old|older|teenage|teen|grown|middle[\s-]aged|elderly"
        r"|single|married|divorced|widowed|straight|gay|bisexual|bi)\s+"
        r"(girl|boy|man|woman|male|female|guy|gal|lady|dude)\b", re.I),
    # as a [gender] / as a young woman
    re.compile(
        r'\bas\s+a\s+'
        r'(girl|boy|man|woman|male|female|guy|gal|lady|dude|gentleman'
        r'|young\s+woman|young\s+man|young\s+lady|young\s+guy'
        r'|grown\s+man|grown\s+woman|grown\s+lady'
        r'|teenage\s+girl|teenage\s+boy|teen\s+girl|teen\s+boy)\b', re.I),
    # being [a] [gender]
    re.compile(
        r'\b(?:being|become|becoming)\s+(?:a\s+)?'
        r'(girl|boy|man|woman|male|female|guy|gal|lady|dude)\b', re.I),
    # [gender] here
    re.compile(
        r'\b(girl|boy|man|woman|male|female|guy|gal|lady|dude|gentleman)\s+here\b', re.I),
    # biologically [male/female]
    re.compile(r'\bbiologically\s+(male|female)\b', re.I),
    # I'm considered / seen as / perceived as [gender]
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:considered|seen\s+as|perceived\s+as|identified\s+as)\s+"
        r"(?:a\s+)?(male|female|man|woman)\b", re.I),
]

# --- Tier 3: TRANS / NB identity ------------------------------------------
TRANS_NB = [
    re.compile(
        r'\b(trans\s*woman|trans\s*man|transwoman|transman'
        r'|trans\s*girl|trans\s*boy|trans\s*guy|trans\s*gal'
        r'|trans\s*lady|trans\s*male|trans\s*female'
        r'|detrans\s*woman|detrans\s*man)\b', re.I),
    re.compile(r'\b(ftm|mtf|f2m|m2f)\b', re.I),
    re.compile(r'\b(amab|afab)\b', re.I),
    re.compile(r'\bassigned\s+(male|female)\s+at\s+birth\b', re.I),
]

# --- Tier 4: PARENTAL / FAMILY self-roles ---------------------------------
PARENTAL = [
    # I am [a] mom/dad/...
    re.compile(
        r"\b(?:i\s+am|i'm|im)\s+(?:a\s+)?"
        r"(mom|mother|mommy|mama|mum|mummy|dad|father|daddy|papa)\b", re.I),
    # as a single mom / stay-at-home dad / ...
    re.compile(
        r'\bas\s+a\s+'
        r'(mom|mother|mommy|mama|mum|mummy|dad|father|daddy|papa'
        r'|single\s+mom|single\s+dad|single\s+mother|single\s+father'
        r'|new\s+mom|new\s+dad|new\s+mother|new\s+father'
        r'|stay[\s-]at[\s-]home\s+mom|stay[\s-]at[\s-]home\s+dad)\b', re.I),
    # Standalone compound roles
    re.compile(
        r'\b(single\s+mom|single\s+dad|single\s+mother|single\s+father'
        r'|new\s+mom|new\s+dad|sahm|sahd'
        r'|stay[\s-]at[\s-]home\s+mom|stay[\s-]at[\s-]home\s+dad)\b', re.I),
]

# --- Tier 5: FELLOW [group] membership ------------------------------------
FELLOW = [
    re.compile(
        r'\bfellow\s+(guys?|girls?|ladies|men|women|boys?|dudes?'
        r'|gals?|gentlemen|brothers?|sisters?'
        r'|moms?|dads?|mothers?|fathers?)\b', re.I),
]

# --- Tier 6: GENDER-SPECIFIC EXPERIENCES (implies female) -----------------
GENDERED_EXP = [
    re.compile(r"\b(?:i\s+am|i'm|im|i\s+was|when\s+i\s+was|being)\s+pregnant\b", re.I),
    re.compile(r'\bmy\s+(?:menstruation|menstrual\s+cycle|pms|period\s+cramps?)\b', re.I),
    re.compile(r'\bmy\s+period(?!\s+of\b)\b', re.I),
    re.compile(r"\b(?:i\s+am|i'm|im)\s+(?:currently\s+)?(?:breastfeeding|nursing)\b", re.I),
    re.compile(r'\bmy\s+pregnancy\b', re.I),
    re.compile(r"\b(?:i\s+)?(?:gave|giving)\s+birth\b", re.I),
    re.compile(r"\b(?:i\s+have|i've|my)\s+postpartum\b", re.I),
]

# --- Tier 7: PARENTHETICAL gender markers ---------------------------------
PAREN_GENDER = [
    # I (F) / me (M)
    re.compile(r'\b(?:i|me|myself)\s*[\(\[]\s*\d*\s*(f|m|nb|male|female|nonbinary)\s*[\)\]]', re.I),
    # Sentence-initial or after punctuation: (F) (M)
    re.compile(r'(?:^|[.!?,;:\n])\s*[\(\[]\s*(f|m|nb|male|female|nonbinary)\s*[\)\]]', re.I | re.M),
]

ALL_TIERS = [AGE_GENDER, SELF_ID, TRANS_NB, PARENTAL, FELLOW, GENDERED_EXP, PAREN_GENDER]
print(f"Pattern tiers: 7  |  Total patterns: {sum(len(t) for t in ALL_TIERS)}")

Pattern tiers: 7  |  Total patterns: 30


In [22]:
def normalize_gender(value):
    """Map raw gender captures to canonical labels."""
    if not value:
        return None
    v = re.sub(r'\s+', ' ', value.strip().lower())

    mapping = {
        # ---- Male ----
        "m": "male", "male": "male", "man": "male", "boy": "male",
        "guy": "male", "dude": "male", "gentleman": "male",
        "dad": "male", "father": "male", "daddy": "male", "papa": "male",
        "single dad": "male", "single father": "male",
        "new dad": "male", "new father": "male", "sahd": "male",
        "stay-at-home dad": "male", "stay at home dad": "male",
        "guys": "male", "men": "male", "boys": "male",
        "dudes": "male", "brothers": "male", "brother": "male",
        "gentlemen": "male", "dads": "male", "fathers": "male",
        "young man": "male", "young guy": "male", "grown man": "male",
        "teenage boy": "male", "teen boy": "male",

        # ---- Female ----
        "f": "female", "female": "female", "woman": "female", "girl": "female",
        "gal": "female", "lady": "female",
        "mom": "female", "mother": "female", "mommy": "female",
        "mama": "female", "mum": "female", "mummy": "female",
        "single mom": "female", "single mother": "female",
        "new mom": "female", "new mother": "female", "sahm": "female",
        "stay-at-home mom": "female", "stay at home mom": "female",
        "girls": "female", "women": "female", "ladies": "female",
        "gals": "female", "sisters": "female", "sister": "female",
        "moms": "female", "mothers": "female", "mamas": "female",
        "young woman": "female", "young lady": "female",
        "grown woman": "female", "grown lady": "female",
        "teenage girl": "female", "teen girl": "female",

        # ---- Nonbinary ----
        "nb": "nonbinary", "nonbinary": "nonbinary", "enby": "nonbinary",

        # ---- Trans ----
        "trans man": "trans_male", "transman": "trans_male",
        "trans boy": "trans_male", "trans guy": "trans_male",
        "trans male": "trans_male",
        "ftm": "trans_male", "f2m": "trans_male",
        "trans woman": "trans_female", "transwoman": "trans_female",
        "trans girl": "trans_female", "trans gal": "trans_female",
        "trans lady": "trans_female", "trans female": "trans_female",
        "mtf": "trans_female", "m2f": "trans_female",
        "detrans man": "detrans_male", "detrans woman": "detrans_female",
        "translady": "trans_female", "transgirl": "trans_female",
        "transguy": "trans_male", "transboy": "trans_male",

        # ---- Assigned at birth ----
        "amab": "amab", "afab": "afab",
    }
    return mapping.get(v, v)


def is_valid_age(age):
    """Check if extracted age is plausible."""
    try:
        return 10 <= int(age) <= 100
    except (ValueError, TypeError):
        return False

In [23]:
TIER_NAMES = ["age_gender", "self_id", "trans_nb", "parental",
              "fellow", "gendered_exp", "paren_gender"]
TIER_CONF  = ["high", "high", "high", "medium",
              "medium", "medium", "medium"]

def extract_gender(text):
    """
    Extract gender self-disclosure from text using 7-tier pattern matching.
    Returns dict with: gender, matched_excerpt, pattern_tier, confidence.
    """
    if pd.isna(text):
        text = ""
    text = str(text)

    # Normalise smart quotes / apostrophes
    text = text.replace('\u2019', "'").replace('\u2018', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')

    # Strip quoted text and Reddit block-quotes (avoid matching others words)
    clean = re.sub(r'"[^"]*"', ' ', text)
    clean = re.sub(r'(?m)^[ \t]*>.*$', ' ', clean)

    result = {"gender": None, "matched_excerpt": None,
              "pattern_tier": None, "confidence": None}

    for tier_pats, tier_name, tier_conf in zip(ALL_TIERS, TIER_NAMES, TIER_CONF):
        for pat in tier_pats:
            m = pat.search(clean)
            if m:
                # Identify which capture group holds the gender token
                gender_raw = None
                for g in m.groups():
                    if g and not g.isdigit():
                        gender_raw = g
                        break

                # Tier 6 (gendered experiences) always implies female
                if tier_name == "gendered_exp":
                    gender_raw = "female"

                if gender_raw:
                    result.update(
                        gender=normalize_gender(gender_raw),
                        matched_excerpt=m.group(0).strip(),
                        pattern_tier=tier_name,
                        confidence=tier_conf,
                    )
                    return result

    return result

In [24]:
extracted = df['text'].progress_apply(extract_gender).apply(pd.Series)
df_annotated = pd.concat([df.reset_index(drop=True), extracted], axis=1)

n_gender = df_annotated['gender'].notna().sum()
print(f"Rows: {len(df_annotated):,}")
print(f"Gender found: {n_gender:,}  ({n_gender/len(df_annotated)*100:.2f}%)")

100%|██████████| 47237/47237 [00:25<00:00, 1847.78it/s]


Rows: 47,237
Gender found: 887  (1.88%)


In [25]:
print("=" * 60)
print("GENDER DISTRIBUTION")
print("=" * 60)
print(df_annotated['gender'].value_counts(dropna=False))

print(f"\n{'=' * 60}\nBY DATASET\n{'=' * 60}")
print(pd.crosstab(df_annotated['dataset'], df_annotated['gender'], margins=True))

print(f"\n{'=' * 60}\nBY LABEL\n{'=' * 60}")
print(pd.crosstab(df_annotated['binary_label'], df_annotated['gender'], margins=True))

print(f"\n{'=' * 60}\nBY PATTERN TIER\n{'=' * 60}")
print(df_annotated['pattern_tier'].value_counts(dropna=False))

print(f"\n{'=' * 60}\nBY CONFIDENCE\n{'=' * 60}")
print(df_annotated['confidence'].value_counts(dropna=False))

GENDER DISTRIBUTION
gender
NaN             46350
male              441
female            399
trans_female       22
trans_male         17
afab                3
amab                3
nonbinary           2
Name: count, dtype: int64

BY DATASET
gender   afab  amab  female  male  nonbinary  trans_female  trans_male  All
dataset                                                                    
cssrs       0     0      26    30          0             3           1   60
mindset     3     3     170    87          2            10          14  289
umd         0     0     203   324          0             9           2  538
All         3     3     399   441          2            22          17  887

BY LABEL
gender        afab  amab  female  male  nonbinary  trans_female  trans_male  \
binary_label                                                                  
0                0     0      45    61          0             3           0   
1                3     3     354   380          2       

In [31]:
# ================================================================
# PROPAGATE GENDER TO ALL POSTS BY IDENTIFIED USERS
# ================================================================
# Strategy: for each user with ≥1 gender-labelled post,
#   1. If unanimous → use that gender
#   2. If conflicting → pick gender from highest-confidence tier (lowest rank)
#   3. If still tied → majority vote
#   Propagate resolved gender to ALL posts by that user_id.

TIER_RANK = {"age_gender": 1, "self_id": 2, "trans_nb": 3, "parental": 4,
             "fellow": 5, "gendered_exp": 6, "paren_gender": 7}

gender_rows = df_annotated[df_annotated['gender'].notna()].copy()
gender_rows['tier_rank'] = gender_rows['pattern_tier'].map(TIER_RANK)

def resolve_gender(group):
    """Resolve a single user's gender from their labelled posts."""
    unique = group['gender'].unique()
    if len(unique) == 1:
        return unique[0]
    # Pick gender from the row with the best (lowest) tier rank
    best_rank = group['tier_rank'].min()
    best_rows = group[group['tier_rank'] == best_rank]
    best_genders = best_rows['gender'].value_counts()
    if len(best_genders) == 1:
        return best_genders.index[0]
    # Fall back to majority vote across all labelled posts
    return group['gender'].value_counts().index[0]

user_gender_map = gender_rows.groupby('user_id').apply(resolve_gender)
user_gender_map.name = 'resolved_gender'
print(f"Resolved gender for {len(user_gender_map)} unique users")

# Check how many were conflicting
n_conflict = (gender_rows.groupby('user_id')['gender'].nunique() > 1).sum()
print(f"  Unanimous: {len(user_gender_map) - n_conflict}")
print(f"  Conflicting (resolved): {n_conflict}")

# ── Propagate to ALL rows ────────────────────────────────────────────────────
df_annotated['propagated_gender'] = df_annotated['user_id'].map(user_gender_map)

# Also track provenance: did the gender come from this row or was it propagated?
df_annotated['gender_source'] = 'none'
df_annotated.loc[df_annotated['gender'].notna(), 'gender_source'] = 'direct'
df_annotated.loc[
    (df_annotated['gender'].isna()) & (df_annotated['propagated_gender'].notna()),
    'gender_source'
] = 'propagated'

n_direct = (df_annotated['gender_source'] == 'direct').sum()
n_propagated = (df_annotated['gender_source'] == 'propagated').sum()
n_total = df_annotated['propagated_gender'].notna().sum()

print(f"\n{'=' * 60}")
print(f"GENDER LABEL COVERAGE AFTER PROPAGATION")
print(f"{'=' * 60}")
print(f"Direct matches:     {n_direct:>6,}")
print(f"Propagated:         {n_propagated:>6,}")
print(f"Total labelled:     {n_total:>6,}  ({n_total/len(df_annotated)*100:.1f}%)")
print(f"Unlabelled:         {len(df_annotated) - n_total:>6,}")

print(f"\nPropagated gender distribution:")
print(df_annotated['propagated_gender'].value_counts(dropna=False).head(10).to_string())

print(f"\nBy dataset:")
labelled_mask = df_annotated['propagated_gender'].notna()
for ds in ['cssrs', 'mindset', 'umd']:
    ds_mask = df_annotated['dataset'] == ds
    n_ds = (ds_mask & labelled_mask).sum()
    tot_ds = ds_mask.sum()
    print(f"  {ds}: {n_ds:,} / {tot_ds:,} ({n_ds/tot_ds*100:.1f}%)")

print(f"\nBy label (binary_label):")
print(pd.crosstab(
    df_annotated.loc[labelled_mask, 'binary_label'],
    df_annotated.loc[labelled_mask, 'propagated_gender'],
    margins=True
))

Resolved gender for 519 unique users
  Unanimous: 478
  Conflicting (resolved): 41

GENDER LABEL COVERAGE AFTER PROPAGATION
Direct matches:        887
Propagated:         15,419
Total labelled:     16,306  (34.5%)
Unlabelled:         30,931

Propagated gender distribution:
propagated_gender
NaN             30931
male            12077
female           3979
trans_female      189
trans_male         53
afab                3
amab                3
nonbinary           2

By dataset:
  cssrs: 1,285 / 6,773 (19.0%)
  mindset: 349 / 19,200 (1.8%)
  umd: 14,672 / 21,264 (69.0%)

By label (binary_label):
propagated_gender  afab  amab  female   male  nonbinary  trans_female  \
binary_label                                                            
0                     0     0     838   1475          0             0   
1                     3     3    3141  10602          2           189   
All                   3     3    3979  12077          2           189   

propagated_gender  trans_male    A

In [8]:
sample = df_annotated.loc[
    df_annotated['gender'].notna(),
    ['text', 'dataset', 'label', 'gender', 'matched_excerpt', 'pattern_tier', 'confidence']
]
print(f"Total gender-annotated: {len(sample)}")
for tier in TIER_NAMES:
    subset = sample[sample['pattern_tier'] == tier]
    if len(subset) > 0:
        print(f"\n--- {tier} ({len(subset)} matches) ---")
        display(subset[['matched_excerpt', 'gender', 'dataset']].head(5))

Total gender-annotated: 887

--- age_gender (477 matches) ---


,matched_excerpt,gender,dataset
342,22 year old male,male,cssrs
355,16 year old boy,male,cssrs
407,16M,male,cssrs
415,50f,female,cssrs
927,21 y/o woman,female,cssrs



--- self_id (229 matches) ---


,matched_excerpt,gender,dataset
20,I am a female,female,cssrs
181,Im a guy,male,cssrs
756,as a girl,female,cssrs
855,being a guy,male,cssrs
1400,Im a guy,male,cssrs



--- trans_nb (49 matches) ---


...

,matched_excerpt,gender,dataset
8772,fellow dad,male,mindset
27093,fellow men,male,umd



--- gendered_exp (79 matches) ---


,matched_excerpt,gender,dataset
1052,being pregnant,female,cssrs
3373,I was pregnant,female,cssrs
3374,I was pregnant,female,cssrs
3430,I was pregnant,female,cssrs
4460,my period,female,cssrs



--- paren_gender (10 matches) ---


,matched_excerpt,gender,dataset
30400,.[M],male,umd
32698,", [M]",male,umd
33267,[F],female,umd
34154,(F),female,umd
35020,[M],male,umd


In [32]:
import os
os.makedirs('data/gender-annotated', exist_ok=True)

# Full annotated dataset (all rows, including propagated gender) as pickle
df_annotated.to_pickle('data/gender-annotated/gender_annotated.pkl')

# Per-dataset CSVs — all rows with a propagated gender label (direct + propagated)
for ds in ['cssrs', 'mindset', 'umd']:
    subset = df_annotated[(df_annotated['dataset'] == ds) & (df_annotated['propagated_gender'].notna())]
    subset.to_csv(f'data/gender-annotated/{ds}_gender_annotated.csv', index=False)
    n_direct = (subset['gender_source'] == 'direct').sum()
    n_prop = (subset['gender_source'] == 'propagated').sum()
    print(f"{ds}: {len(subset):,} rows saved  (direct: {n_direct}, propagated: {n_prop})")

labelled = df_annotated[df_annotated['propagated_gender'].notna()]
print(f"\nTotal labelled: {len(labelled):,} / {len(df_annotated):,} ({len(labelled)/len(df_annotated)*100:.2f}%)")
print("Saved to data/gender-annotated/")

cssrs: 1,285 rows saved  (direct: 60, propagated: 1225)
mindset: 349 rows saved  (direct: 289, propagated: 60)
umd: 14,672 rows saved  (direct: 538, propagated: 14134)

Total labelled: 16,306 / 47,237 (34.52%)
Saved to data/gender-annotated/


In [16]:
inspection = (
    df_annotated[df_annotated['gender'].notna()]
    .sample(n=100, random_state=42)
    .sort_values('dataset')
    [['text', 'dataset', 'label', 'gender', 'matched_excerpt', 'pattern_tier', 'confidence']]
)
inspection.to_csv('data/gender_inspection_sample_100.csv', index=False)
print(f"Saved 100-row inspection sample → data/gender_inspection_sample_100.csv")
inspection

Saved 100-row inspection sample → data/gender_inspection_sample_100.csv


,text,dataset,label,gender,matched_excerpt,pattern_tier,confidence
4540,Ive known that I dont identify as 100% female ...,cssrs,suicidal,trans_male,FtM,trans_nb,high
3372,As a single mom with two dependents thats real...,cssrs,suicidal,female,As a single mom,parental,medium
4095,['That medication is a poison that will suck t...,cssrs,suicidal,female,as a single mother,parental,medium
2370,Hello. Im a trans lady named Jess. I dont look...,cssrs,suicidal,trans_female,trans lady,trans_nb,high
17803,"why is it ok for women to slap, punch and thro...",mindset,has_condition,female,being a woman,self_id,high
...,...,...,...,...,...,...,...
34157,"I had an abortion, and I'm struggling with it....",umd,at_risk,female,I was pregnant,gendered_exp,medium
32130,Today the Church Commemorates: Saint Theodore ...,umd,at_risk,male,as a man,self_id,high
39390,"About to start HRT, wanted to ask a quick ques...",umd,no_risk,trans_female,mtf,trans_nb,high
43541,Told my mom I was an atheist today. She asked ...,umd,no_risk,female,as a mother,parental,medium


In [33]:
# ================================================================
# LIWC-READY CSV — ALL posts by gender-identified users
# ================================================================
# Includes every post by a user whose gender was identified in at least one post.
# Uses propagated_gender (resolved via best-tier + majority vote for conflicts).

liwc_df = df_annotated[df_annotated['propagated_gender'].notna()].copy()
liwc_df = liwc_df.reset_index(drop=True)
liwc_df.insert(0, 'row_id', range(1, len(liwc_df) + 1))

# Keep columns useful for LIWC + downstream analysis
liwc_df = liwc_df[['row_id', 'sentence_id', 'user_id', 'text', 'dataset',
                    'label', 'binary_label',
                    'propagated_gender', 'gender_source',
                    'gender', 'matched_excerpt', 'pattern_tier', 'confidence']]

# Rename for clarity downstream
liwc_df = liwc_df.rename(columns={'propagated_gender': 'gender_label'})

# ── Verify no list-format text remains ───────────────────────────────────────
bad = liwc_df['text'].str.startswith("[\'")
if bad.any():
    print(f"⚠ WARNING: {bad.sum()} rows still have list-format text!")
else:
    print(f"✓ All {len(liwc_df)} rows have clean text (no list-format artifacts)")

out_path = 'data/gender-annotated/all_gender_annotated_for_liwc.csv'
liwc_df.to_csv(out_path, index=False)

print(f"\nRows: {len(liwc_df):,}")
print(f"Columns: {list(liwc_df.columns)}")
print(f"\nGender source breakdown:")
print(f"  Direct (self-disclosure in this post): {(liwc_df['gender_source'] == 'direct').sum():,}")
print(f"  Propagated (other post by same user):  {(liwc_df['gender_source'] == 'propagated').sum():,}")
print(f"\nGender breakdown:\n{liwc_df['gender_label'].value_counts().to_string()}")
print(f"\nDataset breakdown:\n{liwc_df['dataset'].value_counts().to_string()}")
print(f"\nLabel breakdown:\n{liwc_df['binary_label'].value_counts().to_string()}")
print(f"\nSaved → {out_path}")

✓ All 16306 rows have clean text (no list-format artifacts)

Rows: 16,306
Columns: ['row_id', 'sentence_id', 'user_id', 'text', 'dataset', 'label', 'binary_label', 'gender_label', 'gender_source', 'gender', 'matched_excerpt', 'pattern_tier', 'confidence']

Gender source breakdown:
  Direct (self-disclosure in this post): 887
  Propagated (other post by same user):  15,419

Gender breakdown:
gender_label
male            12077
female           3979
trans_female      189
trans_male         53
afab                3
amab                3
nonbinary           2

Dataset breakdown:
dataset
umd        14672
cssrs       1285
mindset      349

Label breakdown:
binary_label
1    13993
0     2313

Saved → data/gender-annotated/all_gender_annotated_for_liwc.csv
